### Importing libraries

In [27]:
import pandas as pd

### Load and clean all 3 datasets


Loading cases, testing, and vaccine data. Standard first pass: strip whitespace
from column names, fix date parsing, and align state names across all three
files.

In [14]:
# Loading the csv files into a dataframe

cases = pd.read_csv("covid_19_india.csv")
testing = pd.read_csv("StatewiseTestingDetails.csv")
vaccine = pd.read_csv("covid_vaccine_statewise.csv")

# Strip whitespace from column names across all 3 files
for df in [cases, testing, vaccine]:
    df.columns = df.columns.str.strip()

In [15]:
print(vaccine.head(5))
print(cases.head(5))
print(testing.head(5))

   Updated On  State  Total Doses Administered  Sessions    Sites  \
0  16/01/2021  India                   48276.0    3455.0   2957.0   
1  17/01/2021  India                   58604.0    8532.0   4954.0   
2  18/01/2021  India                   99449.0   13611.0   6583.0   
3  19/01/2021  India                  195525.0   17855.0   7951.0   
4  20/01/2021  India                  251280.0   25472.0  10504.0   

   First Dose Administered  Second Dose Administered  \
0                  48276.0                       0.0   
1                  58604.0                       0.0   
2                  99449.0                       0.0   
3                 195525.0                       0.0   
4                 251280.0                       0.0   

   Male (Doses Administered)  Female (Doses Administered)  \
0                        NaN                          NaN   
1                        NaN                          NaN   
2                        NaN                          NaN   
3   

Few issues stand out immediately
1. In vaccine dataset, 'State' is being filled with 'India' instead of the state name.
2. testing dataset has mixed date formats

In [16]:
# Fix date columns
cases['Date'] = pd.to_datetime(cases['Date'], format='%Y-%m-%d')
testing['Date'] = pd.to_datetime(testing['Date'], format='mixed', dayfirst=True)
vaccine['Updated On'] = pd.to_datetime(vaccine['Updated On'], format='%d/%m/%Y')

# Standardize state column name
cases = cases.rename(columns={'State/UnionTerritory': 'State'})

# Remove national aggregate row from vaccine data
vaccine = vaccine[vaccine['State'] != 'India']

#### State name consistency check
Checking whether the same states are named identically across all 3 files
— this matters a lot later for merging and for the geopandas choropleth,
where state names must exactly match the shapefile.

In [18]:
print("=== CASES ===")
print(sorted(cases['State'].unique()))

print("\n=== TESTING ===")
print(sorted(testing['State'].unique()))

print("\n=== VACCINE ===")
print(sorted(vaccine['State'].unique()))

=== CASES ===
['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Bihar****', 'Cases being reassigned to states', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Dadra and Nagar Haveli and Daman and Diu', 'Daman & Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Himanchal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karanataka', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Madhya Pradesh***', 'Maharashtra', 'Maharashtra***', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Telengana', 'Tripura', 'Unassigned', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

=== TESTING ===
['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand'

In [19]:
# Drop non-state placeholder rows entirely
cases = cases[~cases['State'].isin(['Cases being reassigned to states', 'Unassigned'])]

# Strip trailing asterisks (footnote markers from source)
cases['State'] = cases['State'].str.rstrip('*')

# Fix misspellings + align the Daman&Diu / Dadra&Nagar Haveli merger naming
state_name_fixes = {
    'Telengana': 'Telangana',
    'Karanataka': 'Karnataka',
    'Himanchal Pradesh': 'Himachal Pradesh',
    'Daman & Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'Dadra and Nagar Haveli': 'Dadra and Nagar Haveli and Daman and Diu',
}
cases['State'] = cases['State'].replace(state_name_fixes)

In [20]:
print(sorted(cases['State'].unique()))
print(len(cases['State'].unique()))

['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
36


In [21]:
# Fix Negative column: one row has a stray blank string instead of NaN,
# which forces the whole column to be read as text instead of numeric
testing['Negative'] = pd.to_numeric(testing['Negative'], errors='coerce')

#### Merge all three datasets into one analysis-ready table

In [22]:
vaccine_merge = vaccine.rename(columns={'Updated On': 'Date'})

merged = cases.merge(testing, on=['State', 'Date'], how='left')
merged = merged.merge(vaccine_merge, on=['State', 'Date'], how='left')

print("\n=== MERGED SHAPE ===")
print(merged.shape)
print(merged[['Date','State','Confirmed','Cured','Deaths','TotalSamples','Positive','Total Individuals Vaccinated']].head(10))

merged.to_csv('covid_merged_clean.csv', index=False)
print("\nSaved covid_merged_clean.csv")


=== MERGED SHAPE ===
(18048, 34)
        Date   State  Confirmed  Cured  Deaths  TotalSamples  Positive  \
0 2020-01-30  Kerala          1      0       0           NaN       NaN   
1 2020-01-31  Kerala          1      0       0           NaN       NaN   
2 2020-02-01  Kerala          2      0       0           NaN       NaN   
3 2020-02-02  Kerala          3      0       0           NaN       NaN   
4 2020-02-03  Kerala          3      0       0           NaN       NaN   
5 2020-02-04  Kerala          3      0       0           NaN       NaN   
6 2020-02-05  Kerala          3      0       0           NaN       NaN   
7 2020-02-06  Kerala          3      0       0           NaN       NaN   
8 2020-02-07  Kerala          3      0       0           NaN       NaN   
9 2020-02-08  Kerala          3      0       0           NaN       NaN   

   Total Individuals Vaccinated  
0                           NaN  
1                           NaN  
2                           NaN  
3              